# Imports

In [1]:
import librosa
import pandas as pd
from typing import Callable, Union, List
import os
from pathlib import Path
import sys
from pprint import pprint

from audio_dataset import RavdessRawData
from Preprocess import audio_to_waveform, trim_silence

# Functions

In [2]:
def extract_audio_statistics(
    audio_paths: List[Path],
    stat_func: Callable[[Path], dict]
) -> pd.DataFrame:
    """
    Generate a DataFrame with statistics for each audio file.

    Each row corresponds to an audio file.
    Columns include the file path and attributes returned by `stat_func`.

    Parameters:
    - audio_paths: List of audio file paths.
    - stat_func: A function that takes a path and returns a dict of stats.

    Returns:
    - pd.DataFrame with one row per file and one column per attribute.
    """
    records = []
    for path in audio_paths:
        stats = stat_func(path)
        stats["path"] = str(path)
        records.append(stats)
    return pd.DataFrame(records)


def no_silence_duration_stat(path: Path) -> dict:
    """
    function to extract duration of the no-silence part of an audio file. meaning the duration after trimming silence from the start and end of the audio file.
    """
    # Here you would implement the logic to get the duration of the audio file.
    waveform, sample_rate = audio_to_waveform(path)
    trimmed_waveform = trim_silence(waveform)
    duration = librosa.get_duration(y=trimmed_waveform, sr=sample_rate)
    return {"duration": duration} 

# Change Working Dir To the Project Working Dir

In [7]:
# change the dir to the grandparent directory of the current working directory
import ipynbname
import os

note_dir = ipynbname.path().parent
grandparent_dir = note_dir.parent.parent
os.chdir(grandparent_dir)

In [8]:
os.getcwd()  # Check the cwd has updated

'c:\\Users\\noams\\Python Projects\\Audio_processing_project'

# Data Examination

## recording's silence analysis

In [18]:
ravdess_raw_data = RavdessRawData()
audio_paths_with_labels = list(ravdess_raw_data.all_data)
audio_paths = [path for path, _ in audio_paths_with_labels]
ravdess_silenced_duraion = extract_audio_statistics(audio_paths, no_silence_duration_stat)
print(ravdess_silenced_duraion.head())  # Display the first few rows of the DataFrame

   duration                                               path
0     1.664  RAVDESS\original_data\Actor_18\03-01-03-02-02-...
1     1.952  RAVDESS\original_data\Actor_12\03-01-03-02-02-...
2     1.888  RAVDESS\original_data\Actor_08\03-01-07-01-02-...
3     1.344  RAVDESS\original_data\Actor_13\03-01-08-02-01-...
4     2.560  RAVDESS\original_data\Actor_03\03-01-06-02-01-...


In [21]:
# show statistics of the audio files
ravdess_silenced_duraion.describe()  # Display the statistics of the DataFrame

,duration
count,1440.000000
mean,1.732832
std,0.349538
min,0.864000
25%,1.504000
50%,1.664000
75%,1.920000
max,3.412437


## TCAV analysis

### RAVDESS dataset

In [34]:
# force re-import of tcav_demo to get the latest changes
import importlib
import tcav_demo
importlib.reload(tcav_demo)

from tcav_demo import get_tcav_per_sample


#? OPTION 1: create the df:
# df_merged = get_tcav_per_sample() # about 10 minutes to run

#? OPTION 2: load tcav_per_sample_with_acc.csv
per_sample_df = pd.read_csv("tcav_per_sample_with_acc.csv")

### df_merged since it's a merge of all_attributes.csv and tcav_per_sample.csv.

In [35]:
display(per_sample_df)
display(per_sample_df.describe())

,path,concept_name,layer_name,positive_percentage,magnitude,cav_acc,true_label,predicted_label,predicted_probability
0,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_constant_thick,module3.blocks.0.conv2,0.0,-0.479544,0.956522,calm,calm,0.999167
1,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_flat_thick,module3.blocks.0.conv2,0.0,-2.054141,0.869565,calm,calm,0.999167
2,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thick,module3.blocks.0.conv2,0.0,-0.572685,0.782609,calm,calm,0.999167
3,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thin,module3.blocks.0.conv2,0.0,-1.140559,0.913043,calm,calm,0.999167
4,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_flat_thick,module3.blocks.0.conv2,0.0,-0.252294,0.695652,calm,calm,0.999167
...,...,...,...,...,...,...,...,...,...
17275,RAVDESS\original_data\Actor_03\03-01-07-02-01-...,short_constant_thick,module3.blocks.0.conv2,1.0,0.739878,0.826087,disgust,disgust,0.999277
17276,RAVDESS\original_data\Actor_03\03-01-07-02-01-...,short_dropping_steep_thick,module3.blocks.0.conv2,1.0,0.700198,0.739130,disgust,disgust,0.999277
17277,RAVDESS\original_data\Actor_03\03-01-07-02-01-...,short_dropping_steep_thin,module3.blocks.0.conv2,1.0,0.733818,0.826087,disgust,disgust,0.999277
17278,RAVDESS\original_data\Actor_03\03-01-07-02-01-...,short_rising_steep_thick,module3.blocks.0.conv2,1.0,1.567698,0.956522,disgust,disgust,0.999277


,positive_percentage,magnitude,cav_acc,predicted_probability
count,17280.000000,17280.000000,17280.000000,17280.000000
mean,0.494618,-0.053511,0.840580,0.944766
std,0.499986,1.188936,0.076003,0.137872
min,0.000000,-4.889296,0.695652,0.233158
25%,0.000000,-0.857114,0.815217,0.994333
50%,0.000000,-0.015965,0.826087,0.998819
75%,1.000000,0.788005,0.880435,0.999190
max,1.000000,4.315073,0.956522,0.999960


#### average among all

In [36]:
# load tcav_per_sample_with_acc.csv
per_sample_df = pd.read_csv(Path(r'tcav_per_sample_with_acc.csv'))

# drop rows whose true_label != predicted_label
per_sample_df = per_sample_df[per_sample_df['true_label'] == per_sample_df['predicted_label']]

# rename 'predicted_label' to 'label'
per_sample_df = per_sample_df.rename(columns={'predicted_label': 'label'})

# drop unnecessary columns
per_sample_df = per_sample_df.drop(columns=['true_label', 'predicted_probability', 'layer_name'])

display(per_sample_df.head(5))
display(per_sample_df.describe())

,path,concept_name,positive_percentage,magnitude,cav_acc,label
0,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_constant_thick,0.0,-0.479544,0.956522,calm
1,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_flat_thick,0.0,-2.054141,0.869565,calm
2,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thick,0.0,-0.572685,0.782609,calm
3,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thin,0.0,-1.140559,0.913043,calm
4,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_flat_thick,0.0,-0.252294,0.695652,calm


,positive_percentage,magnitude,cav_acc
count,16008.000000,16008.000000,16008.000000
mean,0.496377,-0.047172,0.840580
std,0.500002,1.176382,0.076003
min,0.000000,-4.605803,0.695652
25%,0.000000,-0.840683,0.815217
50%,0.000000,-0.011525,0.826087
75%,1.000000,0.787403,0.880435
max,1.000000,4.315073,0.956522


In [43]:
# get the average positive_percentage and magnitude per label and concept_name
df_groupby = per_sample_df.groupby(['label', 'concept_name']).agg({'positive_percentage': 'mean', 'magnitude': 'mean', 'cav_acc': 'mean'}).reset_index()

df_groupby.head(20)
df_groupby.shape

(96, 5)

In [ ]:
df_groupby

In [44]:
df_groupby = df_groupby[(df_groupby['positive_percentage'] >= 0.99) | (df_groupby['positive_percentage'] <= 0.01)]
df_groupby.shape
display(df_groupby)

,label,concept_name,positive_percentage,magnitude,cav_acc
1,angry,long_dropping_flat_thick,0.994652,1.734152,0.869565
3,angry,long_dropping_steep_thin,0.994652,1.329085,0.913043
4,angry,long_rising_flat_thick,0.000000,-0.912051,0.695652
9,angry,short_dropping_steep_thin,0.994652,1.131876,0.826087
13,calm,long_dropping_flat_thick,0.005464,-1.826928,0.869565
24,disgust,long_constant_thick,1.000000,1.242475,0.956522
27,disgust,long_dropping_steep_thin,1.000000,1.518234,0.913043
28,disgust,long_rising_flat_thick,0.994444,0.761245,0.695652
31,disgust,short_constant_thick,1.000000,0.869754,0.826087
34,disgust,short_rising_steep_thick,0.994444,1.398091,0.956522


In [46]:
# give all the rows with surprised and short_rising_steep_thick sorted by magnitude
label_concept_df = per_sample_df[(per_sample_df['label'] == 'surprised') 
                                 & (per_sample_df['concept_name'] == 'short_rising_steep_thick')]
label_concept_df = label_concept_df.sort_values(by='magnitude', ascending=True)
display(label_concept_df)
label_concept_df.shape

,path,concept_name,positive_percentage,magnitude,cav_acc,label
11314,RAVDESS\original_data\Actor_13\03-01-08-01-01-...,short_rising_steep_thick,0.0,-0.117137,0.956522,surprised
15502,RAVDESS\original_data\Actor_11\03-01-08-01-01-...,short_rising_steep_thick,1.0,0.069719,0.956522,surprised
178,RAVDESS\original_data\Actor_09\03-01-08-02-01-...,short_rising_steep_thick,1.0,0.161704,0.956522,surprised
7378,RAVDESS\original_data\Actor_09\03-01-08-01-01-...,short_rising_steep_thick,1.0,0.171181,0.956522,surprised
11386,RAVDESS\original_data\Actor_19\03-01-08-01-02-...,short_rising_steep_thick,1.0,0.211038,0.956522,surprised
...,...,...,...,...,...,...
2134,RAVDESS\original_data\Actor_20\03-01-08-02-01-...,short_rising_steep_thick,1.0,3.485441,0.956522,surprised
9814,RAVDESS\original_data\Actor_03\03-01-08-01-01-...,short_rising_steep_thick,1.0,3.499358,0.956522,surprised
10462,RAVDESS\original_data\Actor_03\03-01-08-01-02-...,short_rising_steep_thick,1.0,3.530239,0.956522,surprised
1138,RAVDESS\original_data\Actor_09\03-01-08-02-02-...,short_rising_steep_thick,1.0,3.673129,0.956522,surprised


(177, 6)

In [47]:
per_sample_df.head(20)

,path,concept_name,positive_percentage,magnitude,cav_acc,label
0,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_constant_thick,0.0,-0.479544,0.956522,calm
1,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_flat_thick,0.0,-2.054141,0.869565,calm
2,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thick,0.0,-0.572685,0.782609,calm
3,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thin,0.0,-1.140559,0.913043,calm
4,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_flat_thick,0.0,-0.252294,0.695652,calm
5,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_steep_thick,0.0,-1.292134,0.869565,calm
6,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_steep_thin,0.0,-0.815764,0.826087,calm
7,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,short_constant_thick,0.0,-0.217394,0.826087,calm
8,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,short_dropping_steep_thick,0.0,-0.479893,0.739130,calm
9,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,short_dropping_steep_thin,0.0,-0.958675,0.826087,calm


#### average among women

In [63]:
# load tcav_per_sample_with_acc.csv
per_sample_df = pd.read_csv(Path(r'tcav_per_sample_with_acc.csv'))

# drop rows whose true_label != predicted_label
per_sample_df = per_sample_df[per_sample_df['true_label'] == per_sample_df['predicted_label']]

# rename 'predicted_label' to 'label'
per_sample_df = per_sample_df.rename(columns={'predicted_label': 'label'})

# drop unnecessary columns
per_sample_df = per_sample_df.drop(columns=['true_label', 'predicted_probability', 'layer_name'])

# remove all rows with men (whose path ends at an odd number)
per_sample_df = per_sample_df[~per_sample_df["path"].str.extract(r"(\d)(?=\.\w+$)")[0].astype(float).mod(2).eq(1)]

display(per_sample_df.head(5))
display(per_sample_df.describe())
print(per_sample_df.shape)

,path,concept_name,positive_percentage,magnitude,cav_acc,label
12,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,long_constant_thick,1.0,0.963829,0.956522,disgust
13,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,long_dropping_flat_thick,1.0,1.250827,0.869565,disgust
14,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,long_dropping_steep_thick,1.0,0.375296,0.782609,disgust
15,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,long_dropping_steep_thin,1.0,1.414820,0.913043,disgust
16,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,long_rising_flat_thick,1.0,0.597090,0.695652,disgust


,positive_percentage,magnitude,cav_acc
count,8124.000000,8124.000000,8124.000000
mean,0.498154,-0.031852,0.840580
std,0.500027,1.205844,0.076005
min,0.000000,-4.399370,0.695652
25%,0.000000,-0.853095,0.815217
50%,0.000000,-0.005300,0.826087
75%,1.000000,0.849214,0.880435
max,1.000000,3.485441,0.956522


(8124, 6)


In [64]:
# get the average positive_percentage and magnitude per label and concept_name
df_groupby = per_sample_df.groupby(['label', 'concept_name']).agg({'positive_percentage': 'mean', 'magnitude': 'mean', 'cav_acc': 'mean'}).reset_index()

display(df_groupby.head(20))
df_groupby.shape

,label,concept_name,positive_percentage,magnitude,cav_acc
0,angry,long_constant_thick,0.819149,0.379567,0.956522
1,angry,long_dropping_flat_thick,0.989362,1.699993,0.869565
2,angry,long_dropping_steep_thick,0.074468,-0.521977,0.782609
3,angry,long_dropping_steep_thin,0.989362,1.330586,0.913043
4,angry,long_rising_flat_thick,0.000000,-0.870802,0.695652
5,angry,long_rising_steep_thick,0.978723,1.235261,0.869565
6,angry,long_rising_steep_thin,0.031915,-0.748963,0.826087
7,angry,short_constant_thick,0.978723,0.830054,0.826087
8,angry,short_dropping_steep_thick,0.159574,-0.285271,0.739130
9,angry,short_dropping_steep_thin,0.989362,1.125401,0.826087


(96, 5)

In [65]:
df_groupby = df_groupby[(df_groupby['positive_percentage'] >= 0.999) | (df_groupby['positive_percentage'] <= 0.001)]
df_groupby.shape
display(df_groupby)

,label,concept_name,positive_percentage,magnitude,cav_acc
4,angry,long_rising_flat_thick,0.0,-0.870802,0.695652
13,calm,long_dropping_flat_thick,0.0,-1.868997,0.869565
24,disgust,long_constant_thick,1.0,1.263740,0.956522
27,disgust,long_dropping_steep_thin,1.0,1.549528,0.913043
28,disgust,long_rising_flat_thick,1.0,0.809023,0.695652
30,disgust,long_rising_steep_thin,1.0,1.156474,0.826087
31,disgust,short_constant_thick,1.0,0.913796,0.826087
34,disgust,short_rising_steep_thick,1.0,1.450036,0.956522
35,disgust,short_rising_steep_thin,1.0,0.855355,0.826087
37,fearful,long_dropping_flat_thick,0.0,-1.773248,0.869565


In [66]:
# give all the rows with surprised and short_rising_steep_thick sorted by magnitude
label_concept_df = per_sample_df[(per_sample_df['label'] == 'surprised') 
                                 & (per_sample_df['concept_name'] == 'short_rising_steep_thick')]
label_concept_df = label_concept_df.sort_values(by='magnitude', ascending=True)
display(label_concept_df)
label_concept_df.shape

,path,concept_name,positive_percentage,magnitude,cav_acc,label
6970,RAVDESS\original_data\Actor_20\03-01-08-01-02-...,short_rising_steep_thick,1.0,0.538428,0.956522,surprised
2230,RAVDESS\original_data\Actor_10\03-01-08-01-01-...,short_rising_steep_thick,1.0,0.599092,0.956522,surprised
15286,RAVDESS\original_data\Actor_06\03-01-08-01-01-...,short_rising_steep_thick,1.0,0.615159,0.956522,surprised
9334,RAVDESS\original_data\Actor_06\03-01-08-02-02-...,short_rising_steep_thick,1.0,0.807847,0.956522,surprised
1930,RAVDESS\original_data\Actor_14\03-01-08-01-02-...,short_rising_steep_thick,1.0,0.827322,0.956522,surprised
...,...,...,...,...,...,...
12910,RAVDESS\original_data\Actor_14\03-01-08-02-02-...,short_rising_steep_thick,1.0,3.167472,0.956522,surprised
7054,RAVDESS\original_data\Actor_24\03-01-08-02-02-...,short_rising_steep_thick,1.0,3.190612,0.956522,surprised
4822,RAVDESS\original_data\Actor_20\03-01-08-01-01-...,short_rising_steep_thick,1.0,3.327538,0.956522,surprised
9514,RAVDESS\original_data\Actor_12\03-01-08-02-01-...,short_rising_steep_thick,1.0,3.358485,0.956522,surprised


(94, 6)

### CREMA-D dataset

In [ ]:
# force re-import of tcav_demo_crema_d to get the latest changes
import importlib
import tcav_demo_crema_d
importlib.reload(tcav_demo_crema_d)

from tcav_demo_crema_d import get_tcav_per_sample


#? OPTION 1: create the df:
# df_merged = get_tcav_per_sample() # about 10 minutes to run

#? OPTION 2: load crema_d_tcav_results_per_sample.csv
per_sample_df = pd.read_csv("crema_d_tcav_results_per_sample.csv")

### df_merged since it's a merge of all_attributes.csv and tcav_per_sample.csv.

In [ ]:
display(per_sample_df.head(5))
display(per_sample_df.describe())

,path,true_label,predicted_label,predicted_probability,concept_name,layer_name,positive_percentage,magnitude
0,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,angry,NaN,long_constant_thick,module3.blocks.0.conv2,1.0,0.547248
1,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,angry,NaN,long_dropping_flat_thick,module3.blocks.0.conv2,1.0,0.215701
2,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,angry,NaN,long_dropping_steep_thick,module3.blocks.0.conv2,0.0,-0.153664
3,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,angry,NaN,long_dropping_steep_thin,module3.blocks.0.conv2,0.0,-0.343864
4,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,angry,NaN,long_rising_flat_thick,module3.blocks.0.conv2,1.0,0.706845


,predicted_probability,positive_percentage,magnitude
count,0.0,89304.000000,89292.000000
mean,NaN,0.474559,0.012933
std,NaN,0.499355,0.759334
min,NaN,0.000000,-3.279677
25%,NaN,0.000000,-0.512262
50%,NaN,0.000000,-0.050605
75%,NaN,1.000000,0.468353
max,NaN,1.000000,4.182953


#### average among all

In [ ]:
# load tcav_per_sample_with_acc.csv
per_sample_df = pd.read_csv("crema_d_tcav_results_per_sample.csv")

# drop rows whose true_label != predicted_label
per_sample_df = per_sample_df[per_sample_df['true_label'] == per_sample_df['predicted_label']]

# rename 'predicted_label' to 'label'
per_sample_df = per_sample_df.rename(columns={'predicted_label': 'label'})

# drop unnecessary columns
per_sample_df = per_sample_df.drop(columns=['true_label', 'predicted_probability', 'layer_name'])

display(per_sample_df.head(5))
display(per_sample_df.describe())

,path,label,concept_name,positive_percentage,magnitude
0,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,long_constant_thick,1.0,0.547248
1,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,long_dropping_flat_thick,1.0,0.215701
2,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,long_dropping_steep_thick,0.0,-0.153664
3,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,long_dropping_steep_thin,0.0,-0.343864
4,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,angry,long_rising_flat_thick,1.0,0.706845


,positive_percentage,magnitude
count,89304.000000,89292.000000
mean,0.474559,0.012933
std,0.499355,0.759334
min,0.000000,-3.279677
25%,0.000000,-0.512262
50%,0.000000,-0.050605
75%,1.000000,0.468353
max,1.000000,4.182953


In [ ]:
# get the average positive_percentage and magnitude per label and concept_name
df_groupby = per_sample_df.groupby(['label', 'concept_name']).agg({'positive_percentage': 'mean', 'magnitude': 'mean'}).reset_index()

df_groupby.head(20)
df_groupby.shape

(72, 4)

In [17]:
df_groupby

,label,concept_name,positive_percentage,magnitude
0,angry,long_constant_thick,0.945712,1.177173
1,angry,long_dropping_flat_thick,0.720692,0.270779
2,angry,long_dropping_steep_thick,0.429583,-0.012141
3,angry,long_dropping_steep_thin,0.039339,-0.506815
4,angry,long_rising_flat_thick,0.952006,1.241597
...,...,...,...,...
67,sad,short_constant_thick,0.148702,-0.239396
68,sad,short_dropping_steep_thick,0.073958,-0.478259
69,sad,short_dropping_steep_thin,0.144768,-0.353554
70,sad,short_rising_steep_thick,0.097561,-0.448810


In [18]:
df_groupby = df_groupby[(df_groupby['positive_percentage'] >= 0.999) | (df_groupby['positive_percentage'] <= 0.001)]
df_groupby.shape
display(df_groupby)

,label,concept_name,positive_percentage,magnitude
24,fearful,long_constant_thick,1.000000,1.618345
29,fearful,long_rising_steep_thick,1.000000,1.493055
33,fearful,short_dropping_steep_thin,0.999213,1.194580
35,fearful,short_rising_steep_thin,0.999213,1.062330
38,happy,long_dropping_steep_thick,0.000787,-0.561562


In [ ]:
# give all the rows with disgust and long_constant_thick sorted by magnitude
per_sample_df = per_sample_df[(per_sample_df['label'] == 'disgust') & (per_sample_df['concept_name'] == 'long_constant_thick')]
per_sample_df = per_sample_df.sort_values(by='magnitude', ascending=False)
display(per_sample_df.head(20))
per_sample_df.shape

,path,label,concept_name,positive_percentage,magnitude
32184,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.440267
69144,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.315096
37728,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.178921
16692,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.175305
5040,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.130638
8208,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.118338
39912,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.090199
9804,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.071663
15696,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.066352
11520,C:\Users\User\Documents\ARIEL\4th_Year\Audio_p...,disgust,long_constant_thick,1.0,0.026467


(1271, 5)

In [ ]:
per_sample_df.head(20)

,path,label,concept_name,positive_percentage,magnitude
16332,RAVDESS\original_data\Actor_07\03-01-07-01-02-...,disgust,long_constant_thick,1.0,3.202342
1356,RAVDESS\original_data\Actor_09\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.943556
12084,RAVDESS\original_data\Actor_02\03-01-07-01-01-...,disgust,long_constant_thick,1.0,2.838130
15828,RAVDESS\original_data\Actor_04\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.547365
972,RAVDESS\original_data\Actor_06\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.505548
6264,RAVDESS\original_data\Actor_24\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.488054
5244,RAVDESS\original_data\Actor_04\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.264716
3624,RAVDESS\original_data\Actor_24\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.220698
13560,RAVDESS\original_data\Actor_15\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.152404
8988,RAVDESS\original_data\Actor_17\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.127205
